# 04 — Test market-data (WebSocket bridge end-to-end)

Smoke test du container `fxvol-market-data` — étape 4/5. Valide la **chaîne complète** depuis l'engine jusqu'à un client WebSocket type frontend :

```
engine market-data
    │ publish_tick → channel ticks (Redis pub/sub)
    ▼
api/ws/redis_bridge.py — SUBSCRIBE ticks → broadcast WS
    │ forward via /ws/ticks
    ▼
nginx — proxy_pass WebSocket sur /ws/ → api:8000
    │ exposé localhost:80
    ▼
Ce notebook (websocket-client) — connect ws://localhost/ws/ticks
```

Si ce test passe → on a la **preuve que le frontend pourra recevoir des ticks**. Le notebook 05 (resilience) sera le dernier de la série market-data ; ensuite tout le pipeline est validé pour passer au container `frontend`.

**Couvre** :
1. Container `fxvol-api` running + healthy (gate, sinon le bridge ne tourne pas)
2. Container `fxvol-nginx` running + healthy (gate, sinon le proxy ne route pas)
3. Connect WebSocket `ws://localhost/ws/ticks` réussit (handshake HTTP 101 Switching Protocols)
4. Réception ≥ 5 messages JSON valides en 5s (preuve que le bridge forward)
5. Schéma identique au pub/sub Redis : `symbol`, `bid`, `ask`, `mid`, `timestamp`
6. Le contenu des messages WS = celui des messages Redis (le bridge ne transforme pas)

**Préreq** :
- Notebooks 01-03 verts
- Stack complète UP : `docker compose --profile engines --profile ib up -d` (ou `start_stack.ps1`)
- `pip install websocket-client` (à ajouter à `requirements.txt` si pas déjà là)

**Référence** : `src/api/routers/ws.py` (endpoint `/ws/ticks`), `src/api/ws/redis_bridge.py` (Redis SUBSCRIBE → WS broadcast), `infrastructure/nginx/nginx-dev.conf` (proxy_pass WS).

## Setup — websocket-client + helpers

On utilise `websocket-client` (sync) plutôt que `websockets` (async) pour rester en cohérence avec le pattern des autres smoke notebooks (synchrones).

In [1]:
import json
import subprocess
import time

try:
    from websocket import WebSocket  # noqa: F401  (vérifier que le package est installé)
    import websocket
except ImportError:
    raise RuntimeError(
        "package `websocket-client` non installé.\n"
        "  -> python -m pip install websocket-client"
    )

WS_URL = "ws://localhost/ws/ticks"
EXPECTED_KEYS = {"symbol", "bid", "ask", "mid", "timestamp"}

MIN_MESSAGES = 5
LISTEN_DURATION_S = 5.0
RECV_TIMEOUT_S = 1.0

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

def docker_inspect(container, fmt):
    out = subprocess.run(
        ["docker", "inspect", "-f", fmt, container],
        capture_output=True, text=True,
    )
    return out.stdout.strip()

print(f"target = {WS_URL}")

target = ws://localhost/ws/ticks


## 1. Gate — `fxvol-api` running + healthy

Le bridge Redis→WS vit dans le container `api`. Si `api` est down, le `/ws/ticks` ne répond pas.

**Si FAIL** :
- `<not found>` → stack pas démarrée avec le profile par défaut. `docker compose up -d api`
- `unhealthy` → vérifier `docker logs fxvol-api --tail 50`. Causes habituelles : Postgres/Redis pas joignables, alembic upgrade qui plante, code bug au boot.

In [2]:
print("== 1. fxvol-api running + healthy ==")

state = docker_inspect("fxvol-api", "{{.State.Status}}")
record("fxvol-api state running", state == "running", state or "<not found>")

health = docker_inspect("fxvol-api", "{{.State.Health.Status}}")
record("fxvol-api healthcheck", health == "healthy", health or "<no healthcheck>")

== 1. fxvol-api running + healthy ==
  [OK  ] fxvol-api state running  | running
  [OK  ] fxvol-api healthcheck  | healthy


## 2. Gate — `fxvol-nginx` running + healthy

Nginx proxy_pass `/ws/` → `api_upstream`. Si nginx est down, on ne peut pas joindre l'api depuis l'host (l'api elle-même n'expose pas de port externe direct, c'est voulu pour la sécurité).

In [3]:
print("== 2. fxvol-nginx running + healthy ==")

state = docker_inspect("fxvol-nginx", "{{.State.Status}}")
record("fxvol-nginx state running", state == "running", state or "<not found>")

health = docker_inspect("fxvol-nginx", "{{.State.Health.Status}}")
record("fxvol-nginx healthcheck", health == "healthy", health or "<no healthcheck>")

== 2. fxvol-nginx running + healthy ==
  [OK  ] fxvol-nginx state running  | running
  [OK  ] fxvol-nginx healthcheck  | healthy


## 3. Connect WebSocket `/ws/ticks`

**Ce que tu dois voir** : le `WebSocket()` se connecte sans timeout. Sous le capot, ça envoie un `GET /ws/ticks` avec `Upgrade: websocket` ; nginx forward, api répond `101 Switching Protocols`, le canal WS est ouvert.

**Si FAIL** :
- `ConnectionError` → nginx down, ou api down
- `WebSocketBadStatusException 502` → nginx OK mais api unreachable depuis l'upstream → `docker compose restart nginx`
- `WebSocketBadStatusException 404` → endpoint mal configuré côté FastAPI (rare)

In [4]:
print("== 3. WebSocket connect ==")

ws = None
try:
    ws = websocket.create_connection(WS_URL, timeout=RECV_TIMEOUT_S)
    record("connect ws://localhost/ws/ticks", ws.connected,
           f"connected = {ws.connected}")
except Exception as e:
    record("connect ws://localhost/ws/ticks", False, f"{type(e).__name__}: {e}")

== 3. WebSocket connect ==
  [OK  ] connect ws://localhost/ws/ticks  | connected = True


## 4. Réception de ≥ 5 messages en 5s

**Ce que tu dois voir** : la même cadence qu'en pub/sub Redis (notebook 03) — ~25 messages max théorique sur 5s à 200ms throttle. ≥ 5 = plancher confortable.

**Si 0 message** :
- WS connecté mais pas de forward → bug `redis_bridge.py` (pas SUBSCRIBE au bon channel)
- engine market-data ne pousse plus → cf. notebooks 02/03 d'abord

In [5]:
print(f"== 4. réception {LISTEN_DURATION_S}s ==")

raw_messages = []
if ws and ws.connected:
    deadline = time.perf_counter() + LISTEN_DURATION_S
    while time.perf_counter() < deadline:
        try:
            data = ws.recv()
            if data:
                raw_messages.append(data)
        except websocket.WebSocketTimeoutException:
            continue  # pas de message dans le timeout, on continue à attendre
        except Exception as e:
            print(f"  [WARN] WS recv error: {type(e).__name__}: {e}")
            break

record(f"≥ {MIN_MESSAGES} messages reçus en {LISTEN_DURATION_S}s",
       len(raw_messages) >= MIN_MESSAGES,
       f"{len(raw_messages)} messages")

== 4. réception 5.0s ==
  [OK  ] ≥ 5 messages reçus en 5.0s  | 26 messages


## 5. JSON valide + schéma identique au pub/sub

**Invariant clé** : le bridge Redis→WS ne **transforme pas** le payload. Donc le format des messages reçus en WS doit être strictement identique à celui qu'on a vu en pub/sub Redis (notebook 03) : 5 clés `symbol`, `bid`, `ask`, `mid`, `timestamp`. Si on observe une différence → le bridge transforme silencieusement, ce qui est un bug.

On valide aussi la cohérence par message (`bid ≤ mid ≤ ask`) — même invariant que le 03, doit tenir aussi après le passage par le bridge.

In [6]:
print("== 5. schéma + cohérence ==")

EPS = 1e-9

if not raw_messages:
    record("schéma + cohérence", False, "skip (cf. §4)")
else:
    parse_errors = 0
    schema_errors = 0
    coherence_errors = 0
    sample = None
    for raw in raw_messages:
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            parse_errors += 1
            continue
        if set(obj.keys()) >= EXPECTED_KEYS:
            if sample is None:
                sample = obj
            bid, mid, ask = obj.get("bid"), obj.get("mid"), obj.get("ask")
            if all(isinstance(v, (int, float)) for v in (bid, mid, ask)):
                if not (bid - EPS <= mid <= ask + EPS):
                    coherence_errors += 1
        else:
            schema_errors += 1
    record("tous les messages JSON-parseables",
           parse_errors == 0,
           f"{parse_errors} erreurs / {len(raw_messages)}")
    record(f"tous ont les clés {sorted(EXPECTED_KEYS)}",
           schema_errors == 0,
           f"{schema_errors} schémas incomplets / {len(raw_messages)}")
    record("bid ≤ mid ≤ ask sur tous les messages",
           coherence_errors == 0,
           f"{coherence_errors} violations / {len(raw_messages)}")
    if sample:
        print(f"  [INFO] sample = {sample}")

== 5. schéma + cohérence ==
  [OK  ] tous les messages JSON-parseables  | 0 erreurs / 26
  [OK  ] tous ont les clés ['ask', 'bid', 'mid', 'symbol', 'timestamp']  | 0 schémas incomplets / 26
  [OK  ] bid ≤ mid ≤ ask sur tous les messages  | 0 violations / 26
  [INFO] sample = {'symbol': 'EURUSD', 'bid': 1.16991, 'ask': 1.16992, 'mid': 1.169915, 'timestamp': '2026-04-28T14:34:56.823862Z'}


## Cleanup

Important de close le WebSocket explicitement — sinon le bridge côté api garde une connexion zombie le temps de son timeout interne, et un re-run du notebook se ferait avec des connexions résiduelles.

In [7]:
if ws:
    try:
        ws.close()
        print("WebSocket closed cleanly.")
    except Exception as e:
        print(f"WS close warning: {type(e).__name__}: {e}")

WebSocket closed cleanly.


## Récap final

In [8]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — chaîne engine → Redis → api → nginx → WS validée bout-à-bout.")
    print("Pass au notebook 05 (resilience) puis on attaque le container frontend.")


LABEL                                                   STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
fxvol-api state running                                 OK      running
fxvol-api healthcheck                                   OK      healthy
fxvol-nginx state running                               OK      running
fxvol-nginx healthcheck                                 OK      healthy
connect ws://localhost/ws/ticks                         OK      connected = True
≥ 5 messages reçus en 5.0s                              OK      26 messages
tous les messages JSON-parseables                       OK      0 erreurs / 26
tous ont les clés ['ask', 'bid', 'mid', 'symbol', 'timestamp'] OK      0 schémas incomplets / 26
bid ≤ mid ≤ ask sur tous les messages                   OK      0 violations / 26
--------------------------------------------------------------------------------------------------------------

9 

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| `fxvol-api unhealthy` | Postgres/Redis pas joignables, ou alembic plante | `docker logs fxvol-api --tail 50` ; vérifier que postgres+redis sont healthy |
| `fxvol-nginx unhealthy` | api upstream pas joignable au boot nginx | `docker compose restart nginx` ; vérifier `infrastructure/nginx/nginx*.conf` |
| `connect WS ConnectionError` | nginx down ou refuse | `docker compose ps` ; restart si besoin |
| `WebSocketBadStatusException 502` | nginx OK mais api ne répond pas sur 8000 | `docker exec fxvol-nginx curl http://api:8000/api/v1/health` (depuis nginx) |
| `connect OK mais 0 message` | bridge `redis_bridge.py` pas SUBSCRIBE au channel `ticks` | grep `CH_TICKS` dans `src/api/ws/redis_bridge.py:_FORWARDED` |
| `connect OK + 0 message` mais notebook 03 reçoit | api/redis pas sur la même DB | comparer `REDIS_URL` env de api vs market-data dans compose |
| `parse_errors > 0` | bridge transforme le payload (ne devrait pas) | inspecter `redis_bridge.py:_serve` ou `connection_manager.broadcast` |
| schéma incomplet | bridge transforme la structure | idem |
| `bid > mid` ou `mid > ask` | bug dans l'engine, déjà couvert par notebook 03 | si fail au 03 aussi → bug engine ; si fail seulement ici → bug bridge (rare) |